In [ ]:
import SimpleITK as sitk
import numpy as np
import os

def bounding_box_center(colon_img):

    # bladder_array = sitk.GetArrayFromImage(bladder_img)  # Shape: [Z, Y, X]
    colon_array = sitk.GetArrayFromImage(colon_img)
    
    # Find the bladder mask Zmax
    colon_indices = np.nonzero(colon_array)
    Z_coords = colon_indices[0]  # Since the array shape is [Z, Y, X]
    
    Zmax = np.max(Z_coords)
    Zmin = np.min(Z_coords)
    Zcut = Zmin + int(0.6 * (Zmax - Zmin))
    Zcentre = int((Zmin + Zcut) / 2)

    # Get the minimum and maximum indices for each dimension
    min_X, max_X = np.min(colon_indices[2]), np.max(colon_indices[2])
    Xcentre = int((min_X + max_X) / 2)
    
    min_Y, max_Y = np.min(colon_indices[1]), np.max(colon_indices[1])
    Ycentre = int((min_Y + max_Y) / 2)

    center = (Xcentre, Ycentre, Zcentre)

    return center

In [ ]:
def center_crop1(image, center, crop_size):
    """
    Perform center cropping on the original image based on the center point.
    :param image: SimpleITK Image object.
    :param center: (x, y, z) coordinates of the center point.
    :param crop_size: (width, height, depth) cropping size.
    :return: Cropped SimpleITK Image.
    """
    # Get image size (x, y, z)
    size = image.GetSize()
    spacing = image.GetSpacing()

    # Calculate start position for cropping
    start = [max(0, min(int(center[i]) - int(crop_size[i]) // 2, int(size[i]) - int(crop_size[i]))) for i in range(3)]

    # Adjust cropping size if it exceeds image dimensions
    adjusted_size = [min(int(crop_size[i]), int(size[i]) - int(start[i])) for i in range(3)]

    # Use SimpleITK RegionOfInterest to crop the image
    cropped_image = sitk.RegionOfInterest(image, adjusted_size, start)

    # Adjust spatial information for the cropped image
    new_origin = [image.GetOrigin()[i] + start[i] * spacing[i] for i in range(3)]
    cropped_image.SetOrigin(new_origin)
    cropped_image.SetSpacing(spacing)
    cropped_image.SetDirection(image.GetDirection())

    return cropped_image

In [ ]:
# Load the original MR image
path = r'/path/to/workspace/classificationmodel_original_sagittal/total_work/resampled'
save_path = r'/path/to/workspace/classificationmodel_original_sagittal/total_work/rectumcentrecrop_correct4'
crop_size = (16,112,112)

for i in os.listdir(path):
    foldername = i # patient name
    file_path = os.path.join(path, i)
    image_path =  os.path.join(file_path, 'resampled_image.nii.gz')
    colon_path =  os.path.join(file_path, 'colon.nii.gz')

    image = sitk.ReadImage(image_path)
    colon_img = sitk.ReadImage(colon_path)

    try:
        center = bounding_box_center(colon_img)
        # rectum_patch = extract_patches(min_Z, max_Z, min_Y, max_Y, min_X, max_X, image)
        print("patient id:", foldername)
        print(f"Center point of the patch: {center}")
    
        # Perform center cropping on the original image
        cropped_img = center_crop1(image, center, crop_size)
    
        # Save the extracted patch
        # new_foldername = foldername 
        save_path_new = os.path.join(save_path, foldername)
        sitk.WriteImage(cropped_img, save_path_new)
        
    except Exception as e:
        print(f"An error occurred: {e}")
        print("patient number:", foldername)
        
    
print('done')